In [1]:
// load the formatting setup, the visualization functions, and the style sheet
#load "../setup.fsx"
#load "../visuals.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The 8-Queens Problem

The <a href="https://en.wikipedia.org/wiki/Eight_queens_puzzle">eight queens puzzle</a> is the problem of placing eight chess queens on a chessboard so that no two queens can capture each other.  In <a href="https://en.wikipedia.org/wiki/Chess">chess</a> a queen can capture another piece if this piece is either
<ol>
    <li>in the same row,</li>
    <li>in the same column, or</li>
    <li>in the same diagonal.</li>
</ol>
The image below shows a queen in row 3, column 4.  All the locations where a piece can be captured by this queen are marked with an arrow.

<img src="queen-captures.png">

We will solve this puzzle by encoding it as a *constraint satisfaction problem*.
- The set of values is $\{1,\cdots,8\}$.  These values are interpreted as columns.
- We will use 8 variables $Q_1$, $\cdots$, $Q_8$.  If $Q_i = j$, then the interpretation 
  is that there is a queen at the position $(i, j)$, i.e. the queen in row $i$ is 
  placed in column $j$. 
- The condition 
  $$ \bigl\{ \texttt{Q}_i \not= \texttt{Q}_j \;\bigm|\; 
      i \in \{1,\cdots,8\} \wedge j \in \{1,\cdots,8\} \wedge j < i \bigr\} 
  $$
  specifies that there is at most one queen in a given column.
- The condition
  $$ \bigl\{ |\texttt{Q}_i - \texttt{Q}_j| \not= |i - j| \;\bigm|\; 
     i \in \{1,\cdots,8\} \wedge j \in \{1,\cdots,8\} \wedge j < i \bigr\}
  $$
  specifies that there is at most one queen in a given diagonal.

We import the backtracking solver.  The notebook `02-Backtracking-Constraint-Solver.ipynb` defines the types `Constraint<'V>` and `CSP<'V>`, the functions `constr` and `different` that create constraints, and the function `solve`.

In [2]:
importNotebook "02-Backtracking-Constraint-Solver.ipynb"

imported 

02-Backtracking-Constraint-Solver.ipynb

The function `queensCSP` takes no arguments.  It returns a CSP coding the 8 queens problem.  The function `abs` computes the absolute value of a number.

In [3]:
let queensCSP () : CSP<int> =
    let S = [ 1 .. 8 ]           // used as indices
    let Q i = sprintf "Q%d" i
    let DifferentColumns   = [ for i in S do
                                 for j in S do
                                   if i < j then different (Q i) (Q j) ]
    let DifferentDiagonals = [ for i in S do
                                 for j in S do
                                   if j < i then
                                     constr (sprintf "|Q%d - Q%d| ≠ %d" i j (i - j)) [ Q i; Q j ]
                                            (fun A -> abs (A[Q i] - A[Q j]) <> i - j) ]
    { Variables   = [ for i in S -> Q i ]
      Values      = [ 1 .. 8 ]
      Constraints = DifferentColumns @ DifferentDiagonals }

Next, we create a *CSP* that encodes the *8-queens-puzzle*.

In [4]:
let CSP = queensCSP ()
CSP

{ Variables = ["Q1"; "Q2"; "Q3"; "Q4"; "Q5"; "Q6"; "Q7"; "Q8"]; Values = [1; 2; 3; 4; 5; 6; 7; 8]; Constraints = [Q1 ≠ Q2; Q1 ≠ Q3; Q1 ≠ Q4; Q1 ≠ Q5; Q1 ≠ Q6; Q1 ≠ Q7; Q1 ≠ Q8; Q2 ≠ Q3; Q2 ≠ Q4; Q2 ≠ Q5; Q2 ≠ Q6; Q2 ≠ Q7; Q2 ≠ Q8; Q3 ≠ Q4; Q3 ≠ Q5; Q3 ≠ Q6; Q3 ≠ Q7; Q3 ≠ Q8; Q4 ≠ Q5; Q4 ≠ Q6; Q4 ≠ Q7; Q4 ≠ Q8; Q5 ≠ Q6; Q5 ≠ Q7; Q5 ≠ Q8; Q6 ≠ Q7; Q6 ≠ Q8; Q7 ≠ Q8; |Q2 - Q1| ≠ 1; |Q3 - Q1| ≠ 2; |Q3 - Q2| ≠ 1; |Q4 - Q1| ≠ 3; |Q4 - Q2| ≠ 2; |Q4 - Q3| ≠ 1; |Q5 - Q1| ≠ 4; |Q5 - Q2| ≠ 3; |Q5 - Q3| ≠ 2; |Q5 - Q4| ≠ 1; |Q6 - Q1| ≠ 5; |Q6 - Q2| ≠ 4; |Q6 - Q3| ≠ 3; |Q6 - Q4| ≠ 2; |Q6 - Q5| ≠ 1; |Q7 - Q1| ≠ 6; |Q7 - Q2| ≠ 5; |Q7 - Q3| ≠ 4; |Q7 - Q4| ≠ 3; |Q7 - Q5| ≠ 2; |Q7 - Q6| ≠ 1; |Q8 - Q1| ≠ 7; |Q8 - Q2| ≠ 6; |Q8 - Q3| ≠ 5; |Q8 - Q4| ≠ 4; |Q8 - Q5| ≠ 3; |Q8 - Q6| ≠ 2; |Q8 - Q7| ≠ 1] }

This *CSP* has 8 variables and 56 constraints.  When we solved the same problem using propositional logic and the *Davis-Putnam algorithm*, we had used 64 propositional variables and 512 clauses.

In [5]:
CSP.Variables.Length, CSP.Constraints.Length

(8, 56)

In [6]:
#!time
let Solution = (solve CSP).Value
Solution

map [("Q1", 1); ("Q2", 5); ("Q3", 8); ("Q4", 6); ("Q5", 3); ("Q6", 7); ("Q7", 2); ("Q8", 4)]

Wall time: 10.0257ms

The function `showSolution` takes three arguments:
* `Solution` is a map that contains a variable assignment that represents a solution to the 8-queens puzzle.
  If $\texttt{Solution[Q}_i\texttt{]} = k$, then the queen in row $i$ is placed in column $k$.
* `n` is the size of the board.
* `width` specifies the size of the board as a CSS length, e.g. a percentage of the width of notebook.

It displays this solution on a chess board.  The function `chessBoard` is defined in the file `visuals.fsx`, which has been loaded in the first cell.

In [7]:
let showSolution (Solution: Map<string, int>) (n: int) (width: string) =
    let queens = [ for row in 1 .. n do
                     match Solution.TryFind (sprintf "Q%d" row) with
                     | Some col -> (row, col)
                     | None     -> () ]
    chessBoard n queens width

In [8]:
showSolution Solution 8 "50%"

♛,,,,,,,
,,,,♛,,,
,,,,,,,♛
,,,,,♛,,
,,♛,,,,,
,,,,,,♛,
,♛,,,,,,
,,,♛,,,,


If we would use a brute force approach that checks permutations, we would have to check $8! = 40\,320$ different configurations.

In [9]:
List.fold (*) 1 [ 1 .. 8 ]

40320